# 2.1 Create MLflow Runs

**Objective:** Train a simple model and create an MLflow run with parameters, metrics, and model artifact.

This notebook is fully independent. It does not depend on helper folders, custom utility modules, or files outside this notebook folder.

In [ ]:


import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


## 2. Create sample dataset and train a model

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X, y = make_classification(n_samples=1000, n_features=10, n_informative=6, n_redundant=2, random_state=42)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[feature_names], df["target"], test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, preds),
    "precision": precision_score(y_test, preds),
    "recall": recall_score(y_test, preds),
    "f1": f1_score(y_test, preds),
}
metrics

## 3. Create and log an MLflow run

In [ ]:
mlflow.set_experiment("independent_02_create_run")

with mlflow.start_run(run_name="random_forest_basic") as run:
    mlflow.log_params({"model_type": "RandomForestClassifier", "n_estimators": 50, "max_depth": 5})
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, artifact_path="model")
    run_id = run.info.run_id

print("Created run id:", run_id)